# NISAR Disaster Detection — demo pipeline
Synthetic GUNW → displacement → coherence change → U-Net → events & risk.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, matplotlib.pyplot as plt
from src.physics import SARPhysics
from src.coherence import CoherenceAnalyzer
from src.model import ModelTrainer
from src.inference import DisasterDetector

## 1. Synthetic GUNW product

In [ ]:
d = SARPhysics.generate_synthetic_gunw(shape=(512, 512), seed=42)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, k, c in zip(ax, ['wrapped_phase', 'amplitude', 'coherence_after'], ['twilight', 'gray', 'inferno']):
    im = a.imshow(d[k], cmap=c); a.set_title(k); a.axis('off'); plt.colorbar(im, ax=a, fraction=0.046)
plt.show()

## 2. Physics: phase → LOS → vertical displacement

In [ ]:
los = SARPhysics.phase_to_displacement(d['unwrapped_phase'], wavelength=0.24)
vert = SARPhysics.compute_los_to_vertical(los, incidence_angle_deg=35.0)
dcoh = CoherenceAnalyzer.compute_coherence_change(d['coherence_before'], d['coherence_after'])
risk = CoherenceAnalyzer.compute_risk_score(vert, dcoh)
print(f'vertical: {vert.min():.1f} .. {vert.max():.1f} mm, max Δγ = {dcoh.max():.2f}')
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
lim = np.abs(vert).max()
for a, arr, t, c, kw in [(ax[0], vert, 'Vertical (mm)', 'RdBu', dict(vmin=-lim, vmax=lim)), (ax[1], dcoh, 'Δ coherence', 'magma', {}), (ax[2], risk, 'Risk score', 'hot', dict(vmin=0, vmax=100))]:
    im = a.imshow(arr, cmap=c, **kw); a.set_title(t); a.axis('off'); plt.colorbar(im, ax=a, fraction=0.046)
plt.show()

## 3. Train U-Net on pseudo-labels

In [ ]:
labels = ModelTrainer.create_pseudo_labels(vert, dcoh)
trainer = ModelTrainer()
stack = np.stack([d['coherence_before'], d['coherence_after'], d['amplitude']])
trainer.prepare_tiles(stack, vert, tile_size=256, overlap=32, labels=labels)
hist = trainer.train(epochs=20, lr=1e-3, streamlit_log=False)
plt.plot(hist); plt.xlabel('epoch'); plt.ylabel('loss'); plt.show()

## 4. Inference with TTA, post-processing, events

In [ ]:
det = DisasterDetector(model=trainer.model, device=trainer.device)
prob = det.ensemble_predict(d['coherence_before'], d['coherence_after'], d['amplitude'])
mask = det.postprocess(prob, min_area_px=50)
events = det.extract_event_polygons(mask, (43.0, 43.4, 76.7, 77.2), vert, dcoh)
for f in events['features']:
    p = f['properties']
    print(CoherenceAnalyzer.level_emoji(p['risk_level']), p['event_id'], p['risk_level'], p['risk_score'], p['area_km2'], 'km²', p['max_subsidence_mm'], 'mm')
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
ax[0].imshow(prob, vmin=0, vmax=1); ax[0].set_title('confidence')
ax[1].imshow(mask, cmap='Reds'); ax[1].set_title('mask')
ax[2].imshow(d['truth_mask'], cmap='Greens'); ax[2].set_title('ground truth')
[a.axis('off') for a in ax]; plt.show()